In [2]:
import numpy as np
import pandas as pd

In [3]:
df = pd.read_csv('./Products.csv')
df.head(3)

,ProductKey,ProductName,Brand,Color,UnitCostUSD,UnitPriceUSD,SubcategoryKey,Subcategory,CategoryKey,Category
0,1,Contoso 512MB MP3 Player E51 Silver,Contoso,Silver,6.62,12.99,101,MP4&MP3,1,Audio
1,2,Contoso 512MB MP3 Player E51 Blue,Contoso,Blue,6.62,12.99,101,MP4&MP3,1,Audio
2,3,Contoso 1G MP3 Player E100 White,Contoso,White,7.40,14.52,101,MP4&MP3,1,Audio


In [4]:
df = df.dropna()

In [6]:
main_feature4m_df = df[['ProductKey','ProductName', 'Brand', 'Color','Subcategory', 'Category']]

In [7]:
main_feature4m_df.dtypes

ProductKey      int64
ProductName    object
Brand          object
Color          object
Subcategory    object
Category       object
dtype: object

In [8]:
import re

main_feature4m_df['Subcategory'] = main_feature4m_df['Subcategory'].str.replace('MP4&MP3', 'MP4 and MP3')
main_feature4m_df['Subcategory'] = main_feature4m_df['Subcategory'].str.replace(' & ', ' and ')
main_feature4m_df['Subcategory'] = main_feature4m_df['Subcategory'].str.replace('Printers, Scanners and Fax', 'Printers Scanners and Fax')

C:\Users\ngchn\AppData\Local\Temp\ipykernel_20484\1586299548.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  main_feature4m_df['Subcategory'] = main_feature4m_df['Subcategory'].str.replace('MP4&MP3', 'MP4 and MP3')
C:\Users\ngchn\AppData\Local\Temp\ipykernel_20484\1586299548.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  main_feature4m_df['Subcategory'] = main_feature4m_df['Subcategory'].str.replace(' & ', ' and ')
C:\Users\ngchn\AppData\Local\Temp\ipykernel_20484\1586299548.py:5: SettingWithCopyWa

In [9]:
main_feature4m_df.head(3)

,ProductKey,ProductName,Brand,Color,Subcategory,Category
0,1,Contoso 512MB MP3 Player E51 Silver,Contoso,Silver,MP4 and MP3,Audio
1,2,Contoso 512MB MP3 Player E51 Blue,Contoso,Blue,MP4 and MP3,Audio
2,3,Contoso 1G MP3 Player E100 White,Contoso,White,MP4 and MP3,Audio


In [10]:
main_feature4m_df['product_details'] = (
    main_feature4m_df['ProductName'] + ' ' + 
    main_feature4m_df['Brand'] + ' ' + 
    main_feature4m_df['Color'] + ' ' + 
    main_feature4m_df['Subcategory'] + ' ' + 
    main_feature4m_df['Category'] 
)

C:\Users\ngchn\AppData\Local\Temp\ipykernel_20484\4168881800.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  main_feature4m_df['product_details'] = (


In [11]:
main_feature4m_df.head(3)

,ProductKey,ProductName,Brand,Color,Subcategory,Category,product_details
0,1,Contoso 512MB MP3 Player E51 Silver,Contoso,Silver,MP4 and MP3,Audio,Contoso 512MB MP3 Player E51 Silver Contoso Si...
1,2,Contoso 512MB MP3 Player E51 Blue,Contoso,Blue,MP4 and MP3,Audio,Contoso 512MB MP3 Player E51 Blue Contoso Blue...
2,3,Contoso 1G MP3 Player E100 White,Contoso,White,MP4 and MP3,Audio,Contoso 1G MP3 Player E100 White Contoso White...


In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vec = TfidfVectorizer(stop_words='english',max_df=0.95,min_df=2,ngram_range=(1,1))
tfidf_matrix = tfidf_vec.fit_transform(main_feature4m_df['product_details'])

In [13]:
# tfidf_matrix is a sparse matrix means it has lot of zeros in it.
tfidf_matrix.shape

(2517, 980)

In [14]:
from sklearn.metrics.pairwise import linear_kernel
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)

def content_based_recommendations(product_id, data, cosine_sim, n=10):
    idx = data[data['ProductKey'] == product_id].index[0]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:n+1]
    product_indices = [i[0] for i in sim_scores]
    return data.iloc[product_indices], sim_scores

input_product_id = int(input("Nhập mã sản phẩm: "))  
recommendations_result, sim_score = content_based_recommendations(input_product_id, main_feature4m_df, cosine_sim)

In [16]:
# Hiển thị kết quả
print("Sản phẩm gợi ý:")
print(recommendations_result[['ProductKey', 'ProductName']]) 

print("\nĐiểm độ tương đồng dự đoán:")
print([round(tup[1], 3) for tup in sim_score])  

Sản phẩm gợi ý:
    ProductKey                                       ProductName
1            2                 Contoso 512MB MP3 Player E51 Blue
3            4                 Contoso 2G MP3 Player E200 Silver
7            8                 Contoso 4G MP3 Player E400 Silver
2            3                  Contoso 1G MP3 Player E100 White
13          14          Contoso 4GB Flash MP3 Player E401 Silver
5            6                  Contoso 2G MP3 Player E200 Black
8            9                  Contoso 4G MP3 Player E400 Black
36          37  Contoso 8GB Clock & Radio MP3 Player X850 Silver
6            7                   Contoso 2G MP3 Player E200 Blue
33          34        Contoso 4GB Portable MP3 Player M450 Black

Điểm độ tương đồng dự đoán:
[0.877, 0.695, 0.684, 0.663, 0.634, 0.6, 0.59, 0.583, 0.574, 0.571]


In [17]:
df_content_base = df[df['ProductKey'].isin(recommendations_result['ProductKey'])]


In [18]:
# save df_content_base to csv
df_content_base.to_csv('df_content_base.csv', index=False)